In [1]:
import pandas as pd
import os 
import sys
sys.path.append('../src')

import numpy as np

from preprocessing import get_dfs, create_static_df, create_notes_df
from tqdm import tqdm
from models import NotesEncoder
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = NotesEncoder(model_name="../models/med-gte-simcse-ger/").to(device)
encoder.model.eval()

/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1957.58it/s]


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inpl

In [5]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
notes = create_notes_df(dfs, filename=None)
notes

Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


,patient_id,transplant_id,rel_days,text,type
0,33,1805,-1,Thorax bed side vom 18.12.2003 19:45 : Herz im...,Röntgen
1,33,1805,0,Farbkodierte Dopplersonographie und Power-Dop...,Sono
2,33,1805,1,Farbkodierte Dopplersonographie und Power-Dopp...,Sono
3,33,1805,2,Farbkodierte Dopplersonographie und Power-Dopp...,Sono
4,33,1805,3,Farbkodierte Dopplersonographie und Power-Dopp...,Sono
...,...,...,...,...,...
367135,37446,14926,194,AZ stabil. Durchfall etwas besser aber noch da...,clinical_assessment
367136,37446,14926,258,AZ gut. Medikation seit letztem Termin unverän...,clinical_assessment
367137,37446,14926,323,"CellCept-Reduktion am 20.04. von 2g auf 1,5g/T...",clinical_assessment
367138,37446,14926,397,Durchfall etwas besser. Corona: inzwischen 4x ...,clinical_assessment


In [6]:
all_patients = set(static_df['patient_id'])

patients_with_notes = set(notes['patient_id'])

patients_no_notes = all_patients - patients_with_notes
patients_no_notes

{1125,
 5350,
 5364,
 5442,
 7007,
 7135,
 7237,
 8894,
 35021,
 35694,
 37928,
 38051,
 38056,
 38081}

In [7]:
# Sanity checks for notes coverage and embedding validity
notes_per_patient = notes.groupby("patient_id").size()
print(f"Patients in notes: {notes_per_patient.shape[0]}")
print(f"Min notes/patient: {notes_per_patient.min()}, Max notes/patient: {notes_per_patient.max()}")

# This is usually empty because groupby().size() only includes patients that appear in notes
patients_with_no_notes = notes_per_patient[notes_per_patient == 0]
print(f"Patients with 0 notes in grouped notes df: {len(patients_with_no_notes)}")

if "embeddings" not in notes.columns:
    print("'embeddings' column not found yet. Run the embedding generation cell first, then reload embeddings into notes.")
    invalid_embeddings = pd.DataFrame()
else:
    def _is_invalid_embedding(x):
        arr = np.asarray(x)
        if arr.size == 0:
            return True
        # Mark as invalid if NaN/Inf is present or vector is effectively all zeros
        if not np.isfinite(arr).all():
            return True
        return np.isclose(np.mean(arr), 0.0)
    
    invalid_embeddings = notes[notes["embeddings"].apply(_is_invalid_embedding)]
    print(f"Invalid embedding rows: {len(invalid_embeddings)}")

invalid_embeddings.head()

Patients in notes: 3419
Min notes/patient: 1, Max notes/patient: 1090
Patients with 0 notes in grouped notes df: 0
'embeddings' column not found yet. Run the embedding generation cell first, then reload embeddings into notes.


""


In [ ]:
# def batch_embed_texts(encoder, texts, batch_size=32):
#     all_embeddings = []
    
#     for i in tqdm(range(0, len(texts), batch_size)):
#         batch = texts[i:i+batch_size]
#         batch_embeddings = encoder(batch).detach().cpu().numpy()
#         if not np.isfinite(batch_embeddings).all():
#             raise ValueError(f"Non-finite embeddings detected in batch starting at index {i}.")
#         all_embeddings.append(batch_embeddings)
    
#     return np.concatenate(all_embeddings, axis=0).astype(np.float32)

# # Direct, simplified embedding generation
# all_embeddings = batch_embed_texts(encoder, notes['text'].tolist())
# np.save("../data/embeddings/emb_med_gte_simcse_en_ger.npy", all_embeddings)
# print("Saved:", "../data/embeddings/emb_med_gte_simcse_en_ger.npy", "shape=", all_embeddings.shape, "dtype=", all_embeddings.dtype)

  0%|          | 3/11474 [00:04<4:46:43,  1.50s/it]


KeyboardInterrupt: 

In [ ]:
loaded_embeddings = np.load("../data/embeddings/emb_med_gte_simcse_en_ger.npy", allow_pickle=True)
notes['embeddings'] = list(loaded_embeddings) 
notes

,patient_id,transplant_id,rel_days,text,type,embeddings
0,33,1805,-1,Thorax bed side vom 18.12.2003 19:45 : Herz im...,Röntgen,"[-0.0024244264, -0.008849075, 0.02347786, 0.00..."
1,33,1805,0,Farbkodierte Dopplersonographie und Power-Dop...,Sono,"[0.0019770786, 0.02120283, -0.0026721784, 0.01..."
2,33,1805,1,Farbkodierte Dopplersonographie und Power-Dopp...,Sono,"[0.00013237733, 0.015311713, -0.0023558734, 0...."
3,33,1805,2,Farbkodierte Dopplersonographie und Power-Dopp...,Sono,"[-0.004943245, 0.016213143, -0.0061095958, 0.0..."
4,33,1805,3,Farbkodierte Dopplersonographie und Power-Dopp...,Sono,"[-0.002963338, 0.01683537, -0.008901045, 0.017..."
...,...,...,...,...,...,...
367135,37446,14926,194,AZ stabil. Durchfall etwas besser aber noch da...,clinical_assessment,"[-0.014935426, -0.004138535, 0.019268034, 0.02..."
367136,37446,14926,258,AZ gut. Medikation seit letztem Termin unverän...,clinical_assessment,"[-0.0039526606, 0.004947439, 0.022554671, 0.01..."
367137,37446,14926,323,"CellCept-Reduktion am 20.04. von 2g auf 1,5g/T...",clinical_assessment,"[-0.014896797, 0.0039872234, -0.0023746334, 0...."
367138,37446,14926,397,Durchfall etwas besser. Corona: inzwischen 4x ...,clinical_assessment,"[-0.0020223162, -0.0069172657, 0.02143311, 0.0..."


In [11]:
BATCH_SIZE = 32

class NotesDataset(torch.utils.data.Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

# Collate function to tokenize and chunk texts
def collate_fn(batch_texts):
    tokenized_texts = []
    text_chunk_mapping = []  # To track which text each chunk belongs to
    for i, text in enumerate(batch_texts):
        tokens = encoder.tokenizer.tokenize(text)
        #chunks = [" ".join(tokens[j:j+512]) for j in range(0, len(tokens), 512)]
        # or just take one chunk
        chunks = tokens[:512]
        tokenized_texts.extend(chunks)
        text_chunk_mapping.extend([i] * len(chunks))
    return tokenized_texts, text_chunk_mapping

# Create DataLoader
dataset = NotesDataset(notes['text'])
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)



In [ ]:
# Process texts in batches and compute embeddings
all_embeddings = [None] * len(notes)  # Placeholder for final embeddings
with torch.no_grad():
    for batch_chunks, text_chunk_mapping in tqdm(dataloader, desc="Computing embeddings in batches"):
        # Get embeddings for all chunks in the batch
        chunk_embeddings = encoder(batch_chunks).detach().cpu()

        # Aggregate embeddings back to their corresponding texts
        for i, text_index in enumerate(text_chunk_mapping):
            if all_embeddings[text_index] is None:
                all_embeddings[text_index] = []
            all_embeddings[text_index].append(chunk_embeddings[i])

# Compute the average embedding for each text
all_embeddings = [
    torch.stack(embeddings).mean(dim=0).numpy() if embeddings else np.zeros(encoder.model.config.hidden_size)
    for embeddings in all_embeddings
]

# Save the embeddings
all_embeddings = np.array(all_embeddings)
np.save("../data/embeddings/emb_gte.npy", all_embeddings)

Computing embeddings in batches:   0%|          | 0/11474 [00:00<?, ?it/s]

Computing embeddings in batches:   0%|          | 4/11474 [00:19<15:54:26,  4.99s/it]


KeyboardInterrupt: 

In [ ]:
loaded_embeddings = np.load("../data/embeddings/emb_gte.npy", allow_pickle=True)
notes['embeddings'] = list(loaded_embeddings) 